# 05 · Оценка и создание CSV
Укажи ту же папку RUN_DIR, в которую 04 сохранил обученные веса.
CHECKPOINT и имя CSV вычисляются из RUN_DIR. Перед оценкой выводятся путь, хеш и эпоха весов.
TTA выбирается по dev; температура — по calibration; holdout используется для отчёта.
Если калибровка уже есть, она загружается с проверкой соответствия checkpoint.
Для CSV включи CREATE_SUBMISSION. Данные Авито не используются для обучения и разметки.


In [ ]:
from pathlib import Path


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
get_ipython().run_line_magic("run", str(ROOT / "notebooks" / "_shared.ipynb"))
print("Проект:", ROOT)


In [ ]:
RUN_DIR = ROOT / "runs/balanced_A_rot7_v1"
CHECKPOINT = RUN_DIR / "best.pt"
MANIFEST = ROOT / "data/processed/hiertext_full/manifest.csv"
TEST_DIR = ROOT / "data/test"
SAMPLE_SUBMISSION = ROOT / "data/sample_submission.csv"
SUBMISSION_PATH = ROOT / "submissions" / f"{RUN_DIR.name}.csv"
CALIBRATE = not (RUN_DIR / "deployment.json").exists()
CREATE_SUBMISSION = False
BENCHMARK = False
BATCH_SIZE = 128
EXPECTED_COUNT = 20000


In [ ]:
def calibrate_and_report(dev, calibration, holdout, run_dir, identity):
    run_dir = Path(run_dir)
    # Режим TTA выбираем до просмотра holdout и до подбора температуры.
    candidates = [False, True] if "logit_rot" in dev else [False]
    tta = min(candidates, key=lambda mode: metrics(dev.y, probabilities(dev, 1.0, mode))["brier"])
    temperature = fit_temperature(calibration, tta)
    p = probabilities(holdout, temperature, tta)
    deploy = {
        "temperature": temperature,
        "tta": tta,
        "identity": identity,
        "selection": "TTA by dev Brier; temperature by calibration Brier; holdout for reporting only",
        "prior_assumption": "Evaluation contains balanced upright/180 pairs; Avito prior is unknown.",
    }
    report = {
        "dev_raw": {str(m): metrics(dev.y, probabilities(dev, 1.0, m)) for m in candidates},
        "calibration_raw": metrics(calibration.y, probabilities(calibration, 1.0, tta)),
        "calibration_fitted": metrics(calibration.y, probabilities(calibration, temperature, tta)),
        "holdout_raw": metrics(holdout.y, probabilities(holdout, 1.0, tta)),
        "holdout_calibrated": metrics(holdout.y, p),
        "holdout_score_ci95": bootstrap_score_ci(holdout, p),
    }
    holdout = holdout.copy()
    holdout["p_180"] = p
    rows = []
    for script, group in holdout.groupby("script"):
        rows.append({"script": script, **metrics(group.y, group.p_180)})
    pd.DataFrame(rows).to_csv(run_dir / "holdout_by_script.csv", index=False)
    holdout.to_csv(run_dir / "holdout_predictions.csv", index=False)
    save_json(run_dir / "deployment.json", deploy)
    save_json(run_dir / "metrics.json", report)
    return deploy, report, holdout


In [ ]:
@torch.inference_mode()
def benchmark(model, paths, cfg, deployment, device, batch_sizes=(1, 128), repeats=30):
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    model.eval()
    results = []
    for size in batch_sizes:
        chosen = [paths[i % len(paths)] for i in range(size)]
        x = torch.stack([letterbox(open_rgb(p), cfg["height"], cfg["width"]) for p in chosen]).to(
            device
        )

        def forward():
            model(x)
            if deployment["tta"]:
                model(torch.flip(x, dims=(-2, -1)))

        for _ in range(10):
            forward()
        sync()
        times = []
        for _ in range(repeats):
            sync()
            t = time.perf_counter()
            forward()
            sync()
            times.append(time.perf_counter() - t)
        results.append(
            {
                "scope": "model_only",
                "batch_size": size,
                "tta": deployment["tta"],
                "device": str(device),
                "cpu_threads": torch.get_num_threads(),
                "median_batch_ms": float(np.median(times) * 1000),
                "p95_batch_ms": float(np.quantile(times, 0.95) * 1000),
                "images_per_second": float(size / np.mean(times)),
            }
        )
        sync()
        t = time.perf_counter()
        predict_images(model, paths, cfg, deployment, device, size)
        sync()
        elapsed = time.perf_counter() - t
        results.append(
            {
                "scope": "decode_preprocess_model_probability_warm_cache",
                "batch_size": size,
                "tta": deployment["tta"],
                "device": str(device),
                "cpu_threads": torch.get_num_threads(),
                "images_per_second": len(paths) / elapsed,
                "total_seconds": elapsed,
            }
        )
    return pd.DataFrame(results)


In [ ]:
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
cfg = checkpoint["config"].copy()
cfg["batch_size"] = BATCH_SIZE
model = build_model(pretrained=False).to(device)  # не скачивает ImageNet-веса
model.load_state_dict(checkpoint["model"])
model.eval()
identity = {
    "checkpoint_sha256": sha256_file(CHECKPOINT),
    "architecture": "mobilenet_v3_small_binary",
    "height": cfg["height"],
    "width": cfg["width"],
    "precision": "fp32",
    "preprocessing": "RGB, no EXIF transpose, symmetric letterbox 127, ImageNet mean/std",
}
print("Устройство:", device)
print("Размер checkpoint, МБ:", CHECKPOINT.stat().st_size / 1e6)
print("Параметров:", sum(p.numel() for p in model.parameters()))

print("Checkpoint:", CHECKPOINT.resolve())
print("SHA256:", identity["checkpoint_sha256"])
print("Лучшая эпоха:", checkpoint["epoch"] + 1, "| Dev Brier:", checkpoint["dev_brier"])
print("CSV:", SUBMISSION_PATH)


In [ ]:
if CALIBRATE:
    if (RUN_DIR / "deployment.json").exists():
        raise FileExistsError("Калибровка уже сохранена. Для инференса поставь CALIBRATE=False.")
    manifest = load_manifest(MANIFEST)
    if sha256_file(MANIFEST) != checkpoint["data_identity"]["manifest_sha256"]:
        raise ValueError("Manifest отличается от того, на котором обучалась модель")
    predictions = {}
    for split in ["dev", "calibration", "holdout"]:
        subset = manifest[manifest.split == split]
        if subset.empty:
            raise ValueError(f"Пустой split: {split}")
        pred = predict_loader(model, make_loader(subset, cfg), device, amp=False, rotated=True)
        predictions[split] = attach_metadata(pred, manifest)
        predictions[split].to_csv(RUN_DIR / f"{split}_logits.csv", index=False)
    deployment, report, holdout = calibrate_and_report(
        predictions["dev"], predictions["calibration"], predictions["holdout"], RUN_DIR, identity
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))
    display(pd.read_csv(RUN_DIR / "holdout_by_script.csv"))
deployment = json.loads((RUN_DIR / "deployment.json").read_text(encoding="utf-8"))
# Калибровку нельзя переносить между разными весами, даже при одинаковой архитектуре.
if deployment["identity"] != identity:
    raise ValueError("Калибровка относится к другим весам или предобработке")
print("Температура:", deployment["temperature"], "| TTA:", deployment["tta"])


Калибровка не обязана улучшать holdout: между источниками бывает сдвиг распределения.
Не откатывай параметры по результату holdout — запиши результат как есть.
Оценка здесь на искусственно сбалансированных 0/180-парах HierText,
поэтому она не является оценкой неизвестного качества на Авито.
Кириллический срез может быть очень маленьким: его размер обязательно учитываем.


Holdout уже просматривался для исходной модели: это повторно используемый набор, а не новая независимая оценка.


In [ ]:
if CALIBRATE:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(5, 5))
    holdout["bin"] = pd.cut(holdout.p_180, bins=np.linspace(0, 1, 11), include_lowest=True)
    curve = holdout.groupby("bin", observed=True).agg(
        predicted=("p_180", "mean"), actual=("y", "mean"), n=("y", "size")
    )
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.plot(curve.predicted, curve.actual, "o-")
    ax.set(xlabel="Средняя p_180", ylabel="Доля y=1", title="Калибровка на holdout")
    ax.grid(True)
    plt.show()
    display(curve)


In [ ]:
if BENCHMARK:
    manifest = load_manifest(MANIFEST)
    rows = manifest[manifest.split == "dev"].head(256)
    speed = benchmark(model, [ROOT / p for p in rows.path], cfg, deployment, device)
    speed.to_csv(RUN_DIR / ("speed_" + device.type + ".csv"), index=False)
    save_json(RUN_DIR / ("speed_environment_" + device.type + ".json"), environment())
    display(speed)


In [ ]:
if CREATE_SUBMISSION:
    ids, paths = test_files(TEST_DIR, SAMPLE_SUBMISSION, EXPECTED_COUNT)
    if device.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    p = predict_images(model, paths, cfg, deployment, device, BATCH_SIZE)
    elapsed = time.perf_counter() - started
    path = write_submission(ids, p, SUBMISSION_PATH, EXPECTED_COUNT)
    save_json(
        path.with_suffix(".metadata.json"),
        {
            "deployment": deployment,
            "environment": environment(),
            "sample_submission_sha256": sha256_file(SAMPLE_SUBMISSION),
            "submission_sha256": sha256_file(path),
            "images": len(ids),
            "seconds_decode_to_probabilities": elapsed,
            "images_per_second": len(ids) / elapsed,
            "batch_size": BATCH_SIZE,
        },
    )
    print("Сохранено:", path, "| Строк:", len(ids), "| Время, сек:", round(elapsed, 2))
else:
    print("Создание CSV выключено: включи CREATE_SUBMISSION, когда будут тест и sample_submission.")
